# 06 - Ground truth: attempted vs happened vs detected

**Question this notebook answers.** Every detection rate in this project divides by a count of attacks that 'landed'. That count came from a flag meaning only that the proxy selected a target field. How far is it from the truth?

**Reads.** `data/processed/effect_oracle.json`

Run top to bottom. Every number below is computed from the raw file named above; nothing is hard-coded.

In [1]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import nbstyle
from nbstyle import BLUE, ORANGE, AQUA, ORDINAL, INK, INK_2, MUTED, CRITICAL
nbstyle.use_style()
PROC = ROOT / "data" / "processed"
RESULTS = ROOT / "results"

def load(name):
    p = PROC / name
    if not p.exists():
        raise FileNotFoundError(
            f"{p} is missing. Raw traces are gitignored; regenerate with the "
            f"command in results/README.md.")
    return json.loads(p.read_text(encoding="utf-8"))

print("reading from", PROC)

reading from F:\UIU\12th\CS\Paper work\repo\data\processed


`attack_landed = any(p.active for p in plans)` is true as soon as the proxy CHANGES AN ARGUMENT. It says nothing about whether an unauthorized effect occurred -- a server that ignored the argument, no-opped, or failed at the application level counts the same as one that really wrote to the attacker's path.

The replacement records five independent fields, none derived from another, decided by an observer reading real state before and after.

In [2]:
rows = load("effect_oracle.json")
hdr = f"{'behaviour':<20}{'mut':>5}{'err':>5}{'authz':>7}{'unauth':>8}{'unk':>5}{'violation':>11}"
print(hdr); print("-"*len(hdr))
for r in rows:
    print(f"{r['behaviour']:<20}"
          f"{str(r['mutation_attempted'])[0]:>5}"
          f"{str(r['protocol_error'])[0]:>5}"
          f"{str(r['authorized_effect_observed'])[0]:>7}"
          f"{str(r['unauthorized_effect_observed'])[0]:>8}"
          f"{str(r['outcome_unknown'])[0]:>5}"
          f"{str(r['detector_violation'])[0]:>11}")

behaviour             mut  err  authz  unauth  unk  violation
-------------------------------------------------------------
honest                  F    F      T       F    F          F
divert_path             T    F      F       T    F          T
substitute_content      T    F      F       T    F          F
silent_noop             F    F      F       F    T          F
refuse                  F    T      F       F    T          F
error_but_writes        T    T      F       T    F          T


## The gap between the old label and the truth

In [3]:
old_label = [r for r in rows if r["mutation_attempted"] and not r["protocol_error"]]
real      = [r for r in rows if r["unauthorized_effect_observed"]]
err_real  = [r for r in rows if r["protocol_error"] and r["unauthorized_effect_observed"]]
print(f"trials the old `attack_landed` counts : {len(old_label)}")
print(f"trials with a REAL unauthorized effect: {len(real)}")
print(f"protocol error, yet a real effect     : {len(err_real)}")
print()
print("That last row is why excluding errored trials -- the correction made")
print("on 2026-09-10 -- is ALSO wrong: it discards a genuine compromise.")
print("Neither the mutation plan nor the protocol status is ground truth.")

trials the old `attack_landed` counts : 2
trials with a REAL unauthorized effect: 3
protocol error, yet a real effect     : 1

That last row is why excluding errored trials -- the correction made
on 2026-09-10 -- is ALSO wrong: it discards a genuine compromise.
Neither the mutation plan nor the protocol status is ground truth.


In [4]:
caught = [r for r in real if r["detector_violation"]]
print(f"detector violations on real compromises: {len(caught)}/{len(real)}")
for r in real:
    mark = "caught " if r["detector_violation"] else "MISSED "
    print(f"  {mark} {r['behaviour']:<20} {r['note']}")

detector violations on real compromises: 2/3
  caught  divert_path          writes elsewhere, claims success
  MISSED  substitute_content   right file, attacker's bytes
  caught  error_but_writes     reports an error AND leaves a forbidden effect


Controlled behaviours, not real servers. This is **methodology validation**: it shows that neither the mutation plan nor the protocol status is ground truth, and that independent world-state observation is required.